In [161]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/llm-classification-finetuning/sample_submission.csv
/kaggle/input/competitions/llm-classification-finetuning/train.csv
/kaggle/input/competitions/llm-classification-finetuning/test.csv


In [162]:
train_df = pd.read_csv('/kaggle/input/competitions/llm-classification-finetuning/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/llm-classification-finetuning/train.csv')

In [163]:
train_df.head(3)

,id,model_a,model_b,prompt,response_a,response_b,winner_model_a,winner_model_b,winner_tie
0,30192,gpt-4-1106-preview,gpt-4-0613,"[""Is it morally right to try to have a certain...","[""The question of whether it is morally right ...","[""As an AI, I don't have personal beliefs or o...",1,0,0
1,53567,koala-13b,gpt-4-0613,"[""What is the difference between marriage lice...","[""A marriage license is a legal document that ...","[""A marriage license and a marriage certificat...",0,1,0
2,65089,gpt-3.5-turbo-0613,mistral-medium,"[""explain function calling. how would you call...","[""Function calling is the process of invoking ...","[""Function calling is the process of invoking ...",0,0,1


In [164]:
mask_bad = (
    (train_df["winner_model_a"] == 1) & ((train_df["winner_model_b"] == 1) | (train_df["winner_tie"] == 1))
)

bad_rows = train_df[mask_bad]

print(len(bad_rows))

0


In [165]:
sum_flag = (
    train_df["winner_model_a"] + train_df["winner_model_b"] + train_df["winner_tie"]
)

mask_zero = (sum_flag == 0)

zero_rows = train_df[mask_zero]

len(zero_rows)


0

In [166]:
y = np.full(len(train_df), 2, dtype=int)

y[train_df["winner_model_a"] == 1] = 0

y[train_df["winner_model_b"] == 1] = 1

In [167]:
def add_features(df):
    df["prompt_len"] = df["prompt"].str.len()
    df["len_a"] = df["response_a"].str.len()
    df["len_b"] = df["response_b"].str.len()
    
    df["len_diff"] = df["len_a"] - df["len_b"]
    
    df["lines_a"] = df["response_a"].str.count("\n") + 1
    df["lines_b"] = df["response_b"].str.count("\n") + 1
    df["lines_diff"] = df["lines_a"] - df["lines_b"]
    
    df["word_count_a"] = df["response_a"].str.split().str.len()
    df["word_count_b"] = df["response_b"].str.split().str.len()
    df["word_count_diff"] = df["word_count_a"] - df["word_count_b"]
    
    df["uniq_word_a"] = (
        df["response_a"]
        .str.split().apply(lambda xs: len(set(xs)) if isinstance(xs, list) else 0)
    )
    
    df["uniq_word_b"] = (
        df["response_b"]
        .str.split().apply(lambda xs: len(set(xs)) if isinstance(xs, list) else 0)
    )
    
    df["uniq_ratio_a"] = df["uniq_word_a"] / df["word_count_a"].replace(0, 1)
    df["uniq_ratio_b"] = df["uniq_word_b"] / df["word_count_b"].replace(0, 1)
    df["uniq_ratio_diff"] = df["uniq_ratio_a"] - df["uniq_ratio_b"]

    return df

In [168]:
train_df = add_features(train_df)

In [169]:
feature_cols = ["prompt_len", "len_a", "len_b", "len_diff", "lines_a", "lines_b", "lines_diff", "word_count_a", "word_count_b", "word_count_diff", "uniq_ratio_a", "uniq_ratio_b", "uniq_ratio_diff"]

X = train_df[feature_cols]

train_df[feature_cols].head()

,prompt_len,len_a,len_b,len_diff,lines_a,lines_b,lines_diff,word_count_a,word_count_b,word_count_diff,uniq_ratio_a,uniq_ratio_b,uniq_ratio_diff
0,165,4538,1206,3332,1,1,0,656,204,452,0.585366,0.647059,-0.061693
1,200,3114,3649,-535,1,1,0,531,571,-40,0.299435,0.457093,-0.157658
2,60,921,1835,-914,1,1,0,138,280,-142,0.630435,0.496429,0.134006
3,87,3182,1562,1620,1,1,0,536,265,271,0.330224,0.460377,-0.130153
4,79,1300,772,528,1,1,0,230,122,108,0.556522,0.663934,-0.107413


In [170]:
from sklearn.model_selection import train_test_split

In [171]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

In [172]:
import lightgbm as lgb

train_data = lgb.Dataset(X_train, label=y_train)
valid_data = lgb.Dataset(X_valid, label=y_valid)

params = {
    "objective": "multiclass",
    "num_class": 3,
    "metric": "multi_logloss",
    "learning_rate": 0.05,
    "num_leaves": 31,
}

model = lgb.train(
    params,
    train_data,
    num_boost_round=500,
    valid_sets=[train_data, valid_data],
    valid_names=["train", "valid"],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50),
        lgb.log_evaluation(period=50),
    ],
)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002754 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2550
[LightGBM] [Info] Number of data points in the train set: 45981, number of used features: 10
[LightGBM] [Info] Start training from score -1.052457
[LightGBM] [Info] Start training from score -1.073231
[LightGBM] [Info] Start training from score -1.174353
Training until validation scores don't improve for 50 rounds
[50]	train's multi_logloss: 1.02283	valid's multi_logloss: 1.03985
[100]	train's multi_logloss: 1.00338	valid's multi_logloss: 1.0388
[150]	train's multi_logloss: 0.986928	valid's multi_logloss: 1.03898
Early stopping, best iteration is:
[133]	train's multi_logloss: 0.992278	valid's multi_logloss: 1.03863


In [173]:
proba_valid = model.predict(X_valid, num_iteration=model.best_iteration)

p_A = proba_valid[:, 0]
p_B = proba_valid[:, 1]
p_Tie = proba_valid[:, 2]

print(proba_valid[:5])
print(p_A[:5], p_B[:5], p_Tie[:5])

[[0.31335794 0.28260121 0.40404085]
 [0.41980778 0.27898088 0.30121134]
 [0.20671793 0.44639812 0.34688395]
 [0.30601564 0.41885474 0.27512962]
 [0.33566761 0.2895778  0.37475459]]
[0.31335794 0.41980778 0.20671793 0.30601564 0.33566761] [0.28260121 0.27898088 0.44639812 0.41885474 0.2895778 ] [0.40404085 0.30121134 0.34688395 0.27512962 0.37475459]


In [174]:
test_df = add_features(test_df)

X_test = test_df[feature_cols]

proba_test = model.predict(X_test)
winner_model_a = proba_test[:, 0]
winner_model_b = proba_test[:, 1]
winner_tie = proba_test[:, 2]

In [175]:
sub = pd.DataFrame({
    "id": test_df["id"],
    "winner_model_a": winner_model_a,
    "winner_model_b": winner_model_b,
    "winner_tie": winner_tie,
})

sub.to_csv("submission.csv", index=False)

In [176]:
sub

,id,winner_model_a,winner_model_b,winner_tie
0,30192,0.793494,0.083281,0.123225
1,53567,0.180309,0.562003,0.257688
2,65089,0.270818,0.474072,0.255110
3,96401,0.465588,0.293906,0.240507
4,198779,0.448490,0.255378,0.296132
...,...,...,...,...
57472,4294656694,0.344877,0.249326,0.405797
57473,4294692063,0.311627,0.327322,0.361050
57474,4294710549,0.719258,0.162159,0.118582
57475,4294899228,0.212446,0.493138,0.294416
